In [1]:
from pathlib import Path
import numpy as np
from scipy import sparse

ROOT = Path("/mnt/g/Research/TUM_connectome/HyperbolicTest")
SRC = ROOT / "results/cell04_orientation_graph"
OUT = ROOT / "results/wholebrain_landmark_robustness"
OUT.mkdir(parents=True, exist_ok=True)

GRAPHS = {}

for name in ("supported", "cone45"):
    with np.load(SRC / f"nodes_{name}.npz") as z:
        ids = z["base_node_id"].copy()
        ijk = z["ijk"].copy()
        largest = z["in_largest"].astype(bool)

    G = sparse.load_npz(SRC / f"edge_lengths_{name}.npz").tocsr()
    assert G.shape == (len(ids), len(ids))
    assert (G != G.T).nnz == 0 and np.all(G.data > 0)

    GRAPHS[name] = dict(G=G, ids=ids, ijk=ijk, largest=largest)
    print(f"{name}: {len(ids):,} nodes, {G.nnz // 2:,} edges")

a, b = GRAPHS["supported"], GRAPHS["cone45"]
ai, bi = np.flatnonzero(a["largest"]), np.flatnonzero(b["largest"])
common, ia, ib = np.intersect1d(
    a["ids"][ai], b["ids"][bi], return_indices=True
)

COMMON_ROWS = {"supported": ai[ia], "cone45": bi[ib]}
COMMON_IJK = a["ijk"][COMMON_ROWS["supported"]]

assert np.array_equal(COMMON_IJK, b["ijk"][COMMON_ROWS["cone45"]])
n_voxels = len(np.unique(COMMON_IJK, axis=0))
assert n_voxels >= 1024, "Insufficient common voxels for 1024 landmarks."

print(f"\nCommon largest-component nodes: {len(common):,}")
print(f"Common distinct voxels: {n_voxels:,}")
print("Ready for multiscale landmark sampling.")


supported: 147,840 nodes, 238,809 edges
cone45: 84,328 nodes, 133,750 edges

Common largest-component nodes: 65,918
Common distinct voxels: 62,630
Ready for multiscale landmark sampling.


In [2]:

# Cell 2: Eight reproducible nested landmark samples.
# Each voxel contributes one common orientation node.

import json
import pandas as pd

SIZES = (64, 128, 192, 256, 384, 512, 768, 1024)
SEEDS = tuple(range(2101, 2109))
N_MAX = max(SIZES)

voxels, inverse = np.unique(COMMON_IJK, axis=0, return_inverse=True)
order = np.argsort(inverse, kind="stable")
counts = np.bincount(inverse)
starts = np.r_[0, np.cumsum(counts)[:-1]]

panels = []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    selected_voxels = rng.choice(len(voxels), N_MAX, replace=False)
    offsets = rng.integers(0, counts[selected_voxels])
    selected = order[starts[selected_voxels] + offsets]

    assert np.array_equal(COMMON_IJK[selected], voxels[selected_voxels])
    assert len(np.unique(COMMON_IJK[selected], axis=0)) == N_MAX

    panel = pd.DataFrame({
        "seed": seed,
        "rank": np.arange(N_MAX),
        "base_node_id": common[selected],
        "supported_row": COMMON_ROWS["supported"][selected],
        "cone45_row": COMMON_ROWS["cone45"][selected],
        "i": COMMON_IJK[selected, 0],
        "j": COMMON_IJK[selected, 1],
        "k": COMMON_IJK[selected, 2],
    })
    panels.append(panel)
    print(f"Seed {seed}: {N_MAX} unique matched voxels")

LANDMARKS = pd.concat(panels, ignore_index=True)
LANDMARKS.to_csv(OUT / "landmark_samples.csv", index=False)

(OUT / "sampling_protocol.json").write_text(json.dumps({
    "sizes": SIZES,
    "seeds": SEEDS,
    "candidate_voxels": len(voxels),
    "sampling": "Uniform distinct common-LCC voxels",
    "node_selection": "One common orientation node per voxel",
    "nesting": "First N positions of each 1024-landmark sample",
    "graphs": ["supported", "cone45"]
}, indent=2))

print(f"\nSaved {len(SEEDS)} samples and {len(SIZES)} nested sizes")
print(f"Total matched landmark records: {len(LANDMARKS):,}")
print(f"Output: {OUT}")


Seed 2101: 1024 unique matched voxels
Seed 2102: 1024 unique matched voxels
Seed 2103: 1024 unique matched voxels
Seed 2104: 1024 unique matched voxels
Seed 2105: 1024 unique matched voxels
Seed 2106: 1024 unique matched voxels
Seed 2107: 1024 unique matched voxels
Seed 2108: 1024 unique matched voxels

Saved 8 samples and 8 nested sizes
Total matched landmark records: 8,192
Output: /mnt/g/Research/TUM_connectome/HyperbolicTest/results/wholebrain_landmark_robustness


In [3]:

# Cell 3: Whole-brain shortest-path distances for all landmark samples.
# Reuse original graphs; save each graph/seed separately for safe resumption.

from scipy.sparse.csgraph import dijkstra
import pandas as pd
import numpy as np
from time import perf_counter

samples = pd.read_csv(OUT / "landmark_samples.csv")
summary = []
BATCH = 16

for graph in ("supported", "cone45"):
    G = GRAPHS[graph]["G"]

    for seed in SEEDS:
        panel = samples[samples.seed == seed].sort_values("rank")
        rows = panel[f"{graph}_row"].to_numpy(dtype=int)
        ids = panel.base_node_id.to_numpy()

        assert len(rows) == max(SIZES)
        assert np.array_equal(GRAPHS[graph]["ids"][rows], ids)

        path = OUT / f"distances_{graph}_{seed}.npz"
        start_time = perf_counter()

        if path.exists():
            with np.load(path) as z:
                D = z["distance_mm"]
                assert np.array_equal(z["base_node_id"], ids)
            print(f"{graph} | seed {seed}: reusing saved distances")
        else:
            n = len(rows)
            D = np.empty((n, n), dtype=np.float64)

            for start in range(0, n, BATCH):
                stop = min(start + BATCH, n)
                distances = dijkstra(
                    G, directed=False, indices=rows[start:stop]
                )
                D[start:stop] = distances[:, rows]

                if stop % 256 == 0 or stop == n:
                    print(f"{graph} | seed {seed}: {stop}/{n} sources")

        assert D.shape == (max(SIZES), max(SIZES))
        assert np.isfinite(D).all()
        assert np.allclose(D, D.T, atol=1e-4, rtol=0)
        assert np.allclose(np.diag(D), 0, atol=1e-6)
        assert np.all(D[np.triu_indices(len(D), 1)] > 0)

        if not path.exists():
            np.savez_compressed(
                path, distance_mm=D, base_node_id=ids,
                graph=graph, seed=seed
            )

        for n in SIZES:
            values = D[:n, :n][np.triu_indices(n, 1)]
            summary.append(dict(
                graph=graph, seed=seed, landmarks=n,
                pairs=len(values),
                median_distance_mm=float(np.median(values)),
                mean_distance_mm=float(np.mean(values)),
                p95_distance_mm=float(np.percentile(values, 95))
            ))

        pd.DataFrame(summary).to_csv(
            OUT / "distance_summary.csv", index=False
        )
        print(f"Completed {graph} | seed {seed} | "
              f"{perf_counter()-start_time:.1f} s\n")

print("All shortest-path calculations completed.")
print(f"Saved {len(summary)} graph/seed/size summaries.")
print(f"Output: {OUT}")


supported | seed 2101: 256/1024 sources
supported | seed 2101: 512/1024 sources
supported | seed 2101: 768/1024 sources
supported | seed 2101: 1024/1024 sources
Completed supported | seed 2101 | 66.4 s

supported | seed 2102: 256/1024 sources
supported | seed 2102: 512/1024 sources
supported | seed 2102: 768/1024 sources
supported | seed 2102: 1024/1024 sources
Completed supported | seed 2102 | 64.6 s

supported | seed 2103: 256/1024 sources
supported | seed 2103: 512/1024 sources
supported | seed 2103: 768/1024 sources
supported | seed 2103: 1024/1024 sources
Completed supported | seed 2103 | 62.8 s

supported | seed 2104: 256/1024 sources
supported | seed 2104: 512/1024 sources
supported | seed 2104: 768/1024 sources
supported | seed 2104: 1024/1024 sources
Completed supported | seed 2104 | 63.9 s

supported | seed 2105: 256/1024 sources
supported | seed 2105: 512/1024 sources
supported | seed 2105: 768/1024 sources
supported | seed 2105: 1024/1024 sources
Completed supported | seed 

In [4]:

# Cell 4: E3/H3 benchmark with restart-safe checkpoints.
# Reuse Cell 3 distances; do not rerun tractography or shortest paths.

import json, hashlib
import numpy as np
import pandas as pd
from time import perf_counter
from scipy.optimize import minimize
from scipy.sparse.csgraph import dijkstra

BENCH_GRAPHS = ("supported",)
BENCH_SEEDS = (2101,)
BENCH_SIZES = (64, 128, 256, 512, 1024)
K_H, MAXITER, STARTS = -4.0, 600, 2

FITDIR = OUT / "multiscale_fits"
FITDIR.mkdir(parents=True, exist_ok=True)

PROTOCOL = dict(
    version="multiscale_v1", dims=3, curvature=K_H,
    fit_fraction=0.8, maxiter=MAXITER, starts=STARTS,
    ftol=1e-10, gtol=1e-6, maxls=40,
    split_seed_rule="990000 + landmark_seed*2048 + n",
    initialization="training-only MDS and random; shared E/H",
    optimization="L-BFGS-B; original metric, bincount gradient"
)

protocol_path = FITDIR / "protocol.json"
if protocol_path.exists():
    assert json.loads(protocol_path.read_text()) == PROTOCOL, "Fit protocol changed."
else:
    protocol_path.write_text(json.dumps(PROTOCOL, indent=2))


def atomic_npz(path, **data):
    temp = path.with_name(path.name + ".tmp")
    with temp.open("wb") as f:
        np.savez_compressed(f, **data)
    temp.replace(path)


def atomic_json(path, data):
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(data, indent=2), encoding="utf-8")
    temp.replace(path)


def pair4(Z, kind, k, i, j):
    A, B = Z[i], Z[j]
    if kind == "euclidean":
        delta = A - B
        dist = np.linalg.norm(delta, axis=1)
        ga = delta / np.maximum(dist[:, None], 1e-12)
        return dist, ga, -ga

    R = 1 / np.sqrt(-k)
    ta = np.sqrt(R*R + np.sum(A*A, axis=1))
    tb = np.sqrt(R*R + np.sum(B*B, axis=1))
    raw = (ta*tb - np.sum(A*B, axis=1)) / (R*R)
    c = np.maximum(raw, 1 + 1e-12)
    factor = (raw > 1 + 1e-12) / (R * np.sqrt(c*c - 1))
    return (
        R*np.arccosh(c),
        ((tb/ta)[:, None]*A - B)*factor[:, None],
        ((ta/tb)[:, None]*B - A)*factor[:, None]
    )


def loss4(flat, n, kind, k, i, j, target):
    Z = flat.reshape(n, 3)
    dist, gi, gj = pair4(Z, kind, k, i, j)
    res = dist - target
    weights = 2.0 * res / len(res)
    grad = np.empty((n, 3))

    # Algebraically equivalent to the original np.add.at gradient.
    for dim in range(3):
        grad[:, dim] = (
            np.bincount(i, weights=weights*gi[:, dim], minlength=n)
            + np.bincount(j, weights=weights*gj[:, dim], minlength=n)
        )
    return float(np.mean(res*res)), grad.ravel()


def starts4(n, i, j, target, split_seed):
    # MDS initialization uses training pairs only.
    W = np.zeros((n, n))
    W[i, j] = W[j, i] = target
    shortest = dijkstra(W, directed=False)
    if not np.isfinite(shortest).all():
        raise ValueError("Training-pair graph is disconnected.")

    J = np.eye(n) - np.ones((n, n)) / n
    vals, vecs = np.linalg.eigh(-0.5 * J @ (shortest**2) @ J)
    x0 = vecs[:, -3:] * np.sqrt(np.maximum(vals[-3:], 0))
    candidates = [
        x0,
        np.random.default_rng(split_seed).normal(size=(n, 3))
    ]
    for X in candidates:
        X -= X.mean(axis=0)
        current = np.linalg.norm(X[i] - X[j], axis=1)
        X *= np.dot(current, target) / max(np.dot(current, current), 1e-14)
    return candidates


def check_ball4(Z, k, i, j, predicted):
    R = 1 / np.sqrt(-k)
    time = np.sqrt(R*R + np.sum(Z*Z, axis=1))
    P = Z / (time[:, None] + R)
    norms = np.sum(P*P, axis=1)
    if np.any(norms >= 1):
        return False, float("inf")

    denom = np.sqrt((1-norms[i])*(1-norms[j]))
    ball = 2*R*np.arcsinh(
        np.linalg.norm(P[i]-P[j], axis=1) / denom
    )
    err = float(np.max(np.abs(ball-predicted)))
    return bool(np.isfinite(err) and err <= 1e-6), err


samples4 = pd.read_csv(OUT / "landmark_samples.csv")
init_key, initial = None, None

for graph in BENCH_GRAPHS:
    for seed in BENCH_SEEDS:
        with np.load(OUT / f"distances_{graph}_{seed}.npz") as z:
            Dfull, ids = z["distance_mm"], z["base_node_id"]

        panel = samples4[samples4.seed == seed].sort_values("rank")
        assert len(ids) == 1024
        assert np.array_equal(ids, panel.base_node_id.to_numpy())

        for n in BENCH_SIZES:
            i, j = np.triu_indices(n, 1)
            raw = Dfull[i, j]
            assert np.isfinite(raw).all() and np.all(raw > 0)

            split_seed = 990000 + seed*2048 + n
            split_path = FITDIR / f"split_s{seed}_n{n}.npz"

            if split_path.exists():
                with np.load(split_path) as z:
                    tr, te = z["train"], z["test"]
            else:
                tr, te = np.split(
                    np.random.default_rng(split_seed).permutation(len(i)),
                    [int(.8 * len(i))]
                )
                atomic_npz(
                    split_path, train=tr, test=te,
                    split_seed=split_seed, landmarks=n
                )

            assert len(tr) + len(te) == len(i)
            assert len(np.unique(np.r_[tr, te])) == len(i)

            scale = float(np.median(raw[tr]))
            target = raw / scale

            fingerprint = hashlib.sha256(
                Dfull[:n, :n].tobytes()
                + tr.tobytes() + te.tobytes()
                + json.dumps(
                    [graph, seed, n, PROTOCOL], sort_keys=True
                ).encode()
            ).hexdigest()

            print(
                f"\n=== {graph} | seed={seed} | n={n} "
                f"| train={len(tr):,} test={len(te):,} ===",
                flush=True
            )

            for kind, k in (("euclidean", 0.0), ("hyperbolic", K_H)):
                stem = f"{graph}_s{seed}_n{n}_{kind}"
                score_path = FITDIR / f"score_{stem}.json"
                checkpoints = [
                    FITDIR / f"fit_{stem}_start{s}.npz"
                    for s in range(STARTS)
                ]

                if score_path.exists() and all(p.exists() for p in checkpoints):
                    row = json.loads(score_path.read_text())
                    assert row["fingerprint"] == fingerprint
                    print(
                        f"[SKIP] {kind}: nRMSE={row['test_nRMSE']:.5f}, "
                        f"fit_ok={row['fit_ok']}",
                        flush=True
                    )
                    continue

                if init_key != (graph, seed, n):
                    print("Preparing training-only MDS and random initialization...", flush=True)
                    init_time = perf_counter()
                    initial = starts4(
                        n, i[tr], j[tr], target[tr], split_seed
                    )
                    init_key = (graph, seed, n)
                    print(
                        f"Initialization: {perf_counter()-init_time:.1f}s",
                        flush=True
                    )

                records = []

                for s, path in enumerate(checkpoints):
                    if path.exists():
                        with np.load(path) as z:
                            assert str(z["fingerprint"]) == fingerprint
                            Z = z["coordinates"]
                            value = float(z["loss"])
                            ok = bool(z["success"])
                            nit = int(z["iterations"])
                            seconds = float(z["seconds"])

                        print(
                            f"  [SKIP] {kind} start {s}: "
                            f"{nit} iters, {seconds:.1f}s, success={ok}",
                            flush=True
                        )
                    else:
                        X = initial[s]

                        if kind == "hyperbolic":
                            R = 1 / np.sqrt(-k)
                            norm = np.linalg.norm(X, axis=1)
                            X = (
                                R*np.sinh(norm/R)[:, None]*X
                                / np.maximum(norm[:, None], 1e-12)
                            )

                        print(
                            f"  [RUN] {kind} start {s+1}/{STARTS}",
                            flush=True
                        )
                        tic = perf_counter()
                        counter = [0]

                        def progress(_):
                            counter[0] += 1
                            if counter[0] % 50 == 0:
                                elapsed = perf_counter() - tic
                                print(
                                    f"      {kind} start {s}: "
                                    f"{counter[0]} iterations, {elapsed:.1f}s",
                                    flush=True
                                )

                        fit = minimize(
                            loss4, X.ravel(),
                            args=(n, kind, k, i[tr], j[tr], target[tr]),
                            jac=True, method="L-BFGS-B",
                            callback=progress,
                            options=dict(
                                maxiter=MAXITER, ftol=1e-10,
                                gtol=1e-6, maxls=40
                            )
                        )

                        seconds = perf_counter() - tic
                        Z = fit.x.reshape(n, 3)
                        value = float(fit.fun)
                        ok = bool(fit.success)
                        nit = int(fit.nit)

                        if not (np.isfinite(Z).all() and np.isfinite(value)):
                            raise RuntimeError(
                                f"Nonfinite {kind} fit; no checkpoint saved."
                            )

                        atomic_npz(
                            path, coordinates=Z, loss=value,
                            success=ok, iterations=nit,
                            seconds=seconds, fingerprint=fingerprint,
                            message=str(fit.message)
                        )
                        print(
                            f"  [DONE] {kind} start {s}: "
                            f"{nit} iters, {seconds:.1f}s, success={ok}",
                            flush=True
                        )

                    records.append((value, ok, nit, seconds, Z))

                # Select by training loss only.
                best = min(records, key=lambda r: r[0])
                pred = pair4(best[4], kind, k, i[te], j[te])[0]

                if kind == "hyperbolic":
                    valid, numerical_error = check_ball4(
                        best[4], k, i[te], j[te], pred
                    )
                else:
                    valid = bool(np.isfinite(pred).all())
                    numerical_error = 0.0

                row = dict(
                    graph=graph, seed=seed, landmarks=n,
                    model=kind, curvature=k,
                    train_pairs=len(tr), test_pairs=len(te),
                    train_scale_mm=scale,
                    train_nRMSE=float(np.sqrt(best[0])),
                    test_nRMSE=float(np.sqrt(np.mean(
                        (pred-target[te])**2
                    ))),
                    fit_ok=bool(best[1] and valid),
                    selected_converged=bool(best[1]),
                    all_starts_converged=bool(all(x[1] for x in records)),
                    numerical_error=float(numerical_error),
                    optimizer_seconds=float(sum(x[3] for x in records)),
                    optimizer_iterations=int(sum(x[2] for x in records)),
                    fingerprint=fingerprint
                )

                atomic_json(score_path, row)

                print(
                    f"[SAVED] {kind}: test nRMSE={row['test_nRMSE']:.5f}, "
                    f"fit_ok={row['fit_ok']}",
                    flush=True
                )

                all_rows = [
                    json.loads(p.read_text())
                    for p in FITDIR.glob("score_*.json")
                ]
                temp_csv = FITDIR / "benchmark_results.csv.tmp"
                pd.DataFrame(all_rows).sort_values(
                    ["graph", "seed", "landmarks", "model"]
                ).to_csv(temp_csv, index=False)
                temp_csv.replace(FITDIR / "benchmark_results.csv")

print("\nBenchmark completed.")
print("Saved:", FITDIR / "benchmark_results.csv")



=== supported | seed=2101 | n=64 | train=1,612 test=404 ===
Preparing training-only MDS and random initialization...
Initialization: 0.0s
  [RUN] euclidean start 1/2
  [DONE] euclidean start 0: 19 iters, 0.0s, success=True
  [RUN] euclidean start 2/2
  [DONE] euclidean start 1: 48 iters, 0.0s, success=True
[SAVED] euclidean: test nRMSE=0.09966, fit_ok=True
  [RUN] hyperbolic start 1/2
  [DONE] hyperbolic start 0: 17 iters, 0.0s, success=True
  [RUN] hyperbolic start 2/2
  [DONE] hyperbolic start 1: 40 iters, 0.0s, success=True
[SAVED] hyperbolic: test nRMSE=0.09031, fit_ok=True

=== supported | seed=2101 | n=128 | train=6,502 test=1,626 ===
Preparing training-only MDS and random initialization...
Initialization: 0.1s
  [RUN] euclidean start 1/2
  [DONE] euclidean start 0: 15 iters, 0.0s, success=True
  [RUN] euclidean start 2/2
  [DONE] euclidean start 1: 39 iters, 0.0s, success=True
[SAVED] euclidean: test nRMSE=0.09044, fit_ok=True
  [RUN] hyperbolic start 1/2
  [DONE] hyperbolic st

In [6]:

# Cell 5: Full 8 x 8 x 2 multiscale robustness analysis.
# Reuse Cell 4 functions, checkpoints and frozen protocol.

import json, hashlib
import numpy as np
import pandas as pd
from time import perf_counter
from scipy.optimize import minimize

assert json.loads((FITDIR / "protocol.json").read_text()) == PROTOCOL
assert len(SEEDS) == len(SIZES) == 8
assert max(SIZES) == 1024

samples = pd.read_csv(OUT / "landmark_samples.csv")
GRAPHS5 = ("supported", "cone45")
TOTAL = len(GRAPHS5) * len(SEEDS) * len(SIZES)


def refresh5():
    files = sorted(FITDIR.glob("score_*.json"))
    scores = pd.DataFrame([json.loads(p.read_text()) for p in files])
    scores = scores.sort_values(["graph", "seed", "landmarks", "model"])
    tmp = FITDIR / "benchmark_results.csv.tmp"
    scores.to_csv(tmp, index=False)
    tmp.replace(FITDIR / "benchmark_results.csv")

    index = ["graph", "seed", "landmarks"]
    e = scores[scores.model == "euclidean"].set_index(index)
    h = scores[scores.model == "hyperbolic"].set_index(index)
    shared = e.index.intersection(h.index)

    rows = []
    for key in shared:
        E, H = e.loc[key], h.loc[key]
        rows.append(dict(
            graph=key[0], seed=key[1], landmarks=key[2],
            E_nRMSE=E.test_nRMSE,
            H_nRMSE=H.test_nRMSE,
            E_minus_H=E.test_nRMSE - H.test_nRMSE,
            relative_reduction=(
                (E.test_nRMSE - H.test_nRMSE) / E.test_nRMSE
            ),
            H_better=bool(H.test_nRMSE < E.test_nRMSE),
            both_fits_ok=bool(E.fit_ok and H.fit_ok)
        ))

    paired = pd.DataFrame(rows).sort_values(index)
    tmp = FITDIR / "paired_results.csv.tmp"
    paired.to_csv(tmp, index=False)
    tmp.replace(FITDIR / "paired_results.csv")

    valid = paired[paired.both_fits_ok]
    summary = valid.groupby(["graph", "landmarks"]).agg(
        repetitions=("seed", "nunique"),
        H_better=("H_better", "sum"),
        mean_E_nRMSE=("E_nRMSE", "mean"),
        mean_H_nRMSE=("H_nRMSE", "mean"),
        mean_reduction=("relative_reduction", "mean"),
        sd_reduction=("relative_reduction", "std")
    ).reset_index()

    tmp = FITDIR / "multiscale_summary.csv.tmp"
    summary.to_csv(tmp, index=False)
    tmp.replace(FITDIR / "multiscale_summary.csv")
    return paired, summary


completed = 0

for graph in GRAPHS5:
    for seed in SEEDS:
        panel = samples[samples.seed == seed].sort_values("rank")
        assert len(panel) == 1024

        with np.load(OUT / f"distances_{graph}_{seed}.npz") as z:
            Dfull = z["distance_mm"].copy()
            ids = z["base_node_id"].copy()

        assert Dfull.shape == (1024, 1024)
        assert np.array_equal(ids, panel.base_node_id.to_numpy())
        assert np.isfinite(Dfull).all()

        for n in SIZES:
            completed += 1
            i, j = np.triu_indices(n, 1)
            raw = Dfull[i, j]

            split_seed = 990000 + seed * 2048 + n
            split_path = FITDIR / f"split_s{seed}_n{n}.npz"
            expected = np.random.default_rng(split_seed).permutation(len(i))
            expected_tr, expected_te = np.split(expected, [int(.8 * len(i))])

            if split_path.exists():
                with np.load(split_path) as z:
                    tr, te = z["train"].copy(), z["test"].copy()
                assert np.array_equal(tr, expected_tr)
                assert np.array_equal(te, expected_te)
            else:
                tr, te = expected_tr, expected_te
                atomic_npz(split_path, train=tr, test=te,
                           split_seed=split_seed, landmarks=n)

            scale = float(np.median(raw[tr]))
            target = raw / scale
            fingerprint = hashlib.sha256(
                Dfull[:n, :n].tobytes()
                + tr.tobytes() + te.tobytes()
                + json.dumps(
                    [graph, seed, n, PROTOCOL], sort_keys=True
                ).encode()
            ).hexdigest()

            print(
                f"\n[{completed}/{TOTAL}] {graph} | "
                f"seed={seed} | landmarks={n} | pairs={len(i):,}",
                flush=True
            )

            initial = None

            for kind, k in (("euclidean", 0.), ("hyperbolic", K_H)):
                stem = f"{graph}_s{seed}_n{n}_{kind}"
                score_path = FITDIR / f"score_{stem}.json"
                checkpoints = [
                    FITDIR / f"fit_{stem}_start{s}.npz"
                    for s in range(STARTS)
                ]

                if score_path.exists():
                    saved = json.loads(score_path.read_text())
                    assert saved["fingerprint"] == fingerprint

                    if all(p.exists() for p in checkpoints):
                        for p in checkpoints:
                            with np.load(p) as z:
                                assert str(z["fingerprint"]) == fingerprint
                                assert z["coordinates"].shape == (n, 3)
                        print(
                            f"  [SKIP] {kind}: "
                            f"nRMSE={saved['test_nRMSE']:.5f}, "
                            f"fit_ok={saved['fit_ok']}",
                            flush=True
                        )
                        continue

                records = []

                for s, path in enumerate(checkpoints):
                    if path.exists():
                        with np.load(path) as z:
                            assert str(z["fingerprint"]) == fingerprint
                            Z = z["coordinates"].copy()
                            value = float(z["loss"])
                            ok = bool(z["success"])
                            nit = int(z["iterations"])
                            seconds = float(z["seconds"])

                        assert Z.shape == (n, 3)
                        print(f"  [SKIP] {kind} start {s}", flush=True)

                    else:
                        if initial is None:
                            tic = perf_counter()
                            initial = starts4(
                                n, i[tr], j[tr], target[tr], split_seed
                            )
                            print(
                                f"  Initialization: "
                                f"{perf_counter()-tic:.1f}s",
                                flush=True
                            )

                        X = initial[s].copy()

                        if kind == "hyperbolic":
                            R = 1 / np.sqrt(-k)
                            norm = np.linalg.norm(X, axis=1)
                            X = (
                                R * np.sinh(norm/R)[:, None] * X
                                / np.maximum(norm[:, None], 1e-12)
                            )

                        print(
                            f"  [RUN] {kind} start {s+1}/{STARTS}",
                            flush=True
                        )
                        tic = perf_counter()
                        counter = [0]

                        def progress(_):
                            counter[0] += 1
                            if counter[0] % 50 == 0:
                                print(
                                    f"    {counter[0]} iterations | "
                                    f"{perf_counter()-tic:.1f}s",
                                    flush=True
                                )

                        fit = minimize(
                            loss4, X.ravel(),
                            args=(n, kind, k, i[tr], j[tr], target[tr]),
                            jac=True, method="L-BFGS-B",
                            callback=progress,
                            options=dict(
                                maxiter=MAXITER, ftol=1e-10,
                                gtol=1e-6, maxls=40
                            )
                        )

                        Z = fit.x.reshape(n, 3)
                        value = float(fit.fun)
                        ok = bool(fit.success)
                        nit = int(fit.nit)
                        seconds = perf_counter() - tic

                        if not np.isfinite(Z).all() or not np.isfinite(value):
                            raise ValueError("Nonfinite fit; checkpoint not saved.")

                        atomic_npz(
                            path, coordinates=Z, loss=value,
                            success=ok, iterations=nit,
                            seconds=seconds, fingerprint=fingerprint,
                            message=str(fit.message)
                        )
                        print(
                            f"  [DONE] {kind} start {s}: "
                            f"{nit} iterations, {seconds:.1f}s, "
                            f"success={ok}",
                            flush=True
                        )

                    records.append((value, ok, nit, seconds, Z))

                best = min(records, key=lambda x: x[0])
                pred = pair4(best[4], kind, k, i[te], j[te])[0]

                if kind == "hyperbolic":
                    numerical_ok, numerical_error = check_ball4(
                        best[4], k, i[te], j[te], pred
                    )
                else:
                    numerical_ok = bool(np.isfinite(pred).all())
                    numerical_error = 0.0

                result = dict(
                    graph=graph, seed=int(seed), landmarks=int(n),
                    model=kind, curvature=float(k),
                    train_pairs=len(tr), test_pairs=len(te),
                    train_scale_mm=scale,
                    train_nRMSE=float(np.sqrt(best[0])),
                    test_nRMSE=float(np.sqrt(np.mean(
                        (pred - target[te])**2
                    ))),
                    fit_ok=bool(best[1] and numerical_ok),
                    selected_converged=bool(best[1]),
                    all_starts_converged=bool(all(r[1] for r in records)),
                    numerical_error=float(numerical_error),
                    optimizer_seconds=float(sum(r[3] for r in records)),
                    optimizer_iterations=int(sum(r[2] for r in records)),
                    fingerprint=fingerprint
                )

                atomic_json(score_path, result)

                print(
                    f"  [SAVED] {kind}: "
                    f"test nRMSE={result['test_nRMSE']:.5f}, "
                    f"fit_ok={result['fit_ok']}",
                    flush=True
                )

                if not result["fit_ok"]:
                    print("  WARNING: Flagged fit; review before interpretation.")

            paired, summary = refresh5()
            current = paired[
                (paired.graph == graph) &
                (paired.seed == seed) &
                (paired.landmarks == n)
            ]

            if len(current) == 1:
                r = current.iloc[0]
                print(
                    f"  E={r.E_nRMSE:.5f} | H={r.H_nRMSE:.5f} | "
                    f"H reduction={100*r.relative_reduction:.2f}% | "
                    f"valid={r.both_fits_ok}",
                    flush=True
                )

print("\nFULL EXPERIMENT FINISHED")
print(f"Paired comparisons: {len(paired)}/{TOTAL}")
print(f"Valid comparisons: {int(paired.both_fits_ok.sum())}/{TOTAL}")
print(f"Hyperbolic wins among valid comparisons: "
      f"{int((paired.H_better & paired.both_fits_ok).sum())}")
print("\n", summary.to_string(index=False))
print("\nSaved to:", FITDIR)



[1/128] supported | seed=2101 | landmarks=64 | pairs=2,016
  [SKIP] euclidean: nRMSE=0.09966, fit_ok=True
  [SKIP] hyperbolic: nRMSE=0.09031, fit_ok=True
  E=0.09966 | H=0.09031 | H reduction=9.38% | valid=True

[2/128] supported | seed=2101 | landmarks=128 | pairs=8,128
  [SKIP] euclidean: nRMSE=0.09044, fit_ok=True
  [SKIP] hyperbolic: nRMSE=0.07619, fit_ok=True
  E=0.09044 | H=0.07619 | H reduction=15.75% | valid=True

[3/128] supported | seed=2101 | landmarks=192 | pairs=18,336
  [SKIP] euclidean: nRMSE=0.08191, fit_ok=True
  [SKIP] hyperbolic: nRMSE=0.07091, fit_ok=True
  E=0.08191 | H=0.07091 | H reduction=13.42% | valid=True

[4/128] supported | seed=2101 | landmarks=256 | pairs=32,640
  [SKIP] euclidean: nRMSE=0.08587, fit_ok=True
  [SKIP] hyperbolic: nRMSE=0.07089, fit_ok=True
  E=0.08587 | H=0.07089 | H reduction=17.44% | valid=True

[5/128] supported | seed=2101 | landmarks=384 | pairs=73,536
  [SKIP] euclidean: nRMSE=0.08507, fit_ok=True
  [SKIP] hyperbolic: nRMSE=0.07039,

In [7]:

# Cell 6: Audit ROI eligibility for multiscale sampling.

import pandas as pd

OLD_ROI = ROOT / "results/cell20_all120_rois_supported/roi_summary.csv"
roi = pd.read_csv(OLD_ROI).sort_values("roi_id")

ROI_SIZES = (128, 256, 512, 768)
available = roi["largest_voxels"].fillna(0).astype(int)

print(f"Candidate ROIs: {len(roi)}\n")

for n in ROI_SIZES:
    count = int((available >= n).sum())
    print(f"{n:4d} landmarks: {count:3d}/120 eligible ROIs")

print(f"\nAll four scales: {(available >= 768).sum()} ROIs")
print(f"Below 128 landmarks: {(available < 128).sum()} ROIs")

audit = roi[["roi_id", "status", "largest_voxels"]].copy()
for n in ROI_SIZES:
    audit[f"eligible_{n}"] = available >= n

audit.to_csv(OUT / "roi_multiscale_eligibility.csv", index=False)
display(audit)


Candidate ROIs: 120

 128 landmarks:  15/120 eligible ROIs
 256 landmarks:   8/120 eligible ROIs
 512 landmarks:   4/120 eligible ROIs
 768 landmarks:   2/120 eligible ROIs

All four scales: 2 ROIs
Below 128 landmarks: 105 ROIs


,roi_id,status,largest_voxels,eligible_128,eligible_256,eligible_512,eligible_768
0,1,ok,90,False,False,False,False
1,2,ok,38,False,False,False,False
2,3,ok,242,True,False,False,False
3,4,ok,113,False,False,False,False
4,5,ok,86,False,False,False,False
...,...,...,...,...,...,...,...
115,116,ok,25,False,False,False,False
116,117,ok,40,False,False,False,False
117,118,insufficient_component,13,False,False,False,False
118,119,insufficient_component,14,False,False,False,False
